# Session 9 Introduction to Deep Learning

**Learning goals:**

- distinguish traditional machine learning from deep learning
- explain why convolutional neural networks are suitable for image data
- distinguish image classification, object detection and image generation
- explain how recurrent neural networks process speech and text sequences
- describe Seq2Seq, attention and their connection to large language models
- compare the roles and workflows of PyTorch and TensorFlow

**Prerequisites:**

- basic Python and NumPy
- an understanding of supervised learning and neural-network fundamentals
- familiarity with classification and model evaluation

Deep learning extends the neural-network ideas introduced earlier in the course. This session asks three connected questions: why deep learning became practical, which architectures suit different data, and how programming frameworks support model development.


## Outline

1. From machine learning to deep learning
2. Deep learning architectures and applications
   - convolutional neural networks for images
   - recurrent neural networks for speech and text
   - Transformers and large language models, including the Seq2Seq-to-attention transition
3. Deep learning programming frameworks: PyTorch and TensorFlow


## 1. From Machine Learning to Deep Learning

### 1.1 Learning features from data

Traditional machine-learning systems often separate **feature engineering** from **model learning**. A practitioner first converts raw data into measurements thought to be useful, then trains a model on those measurements.

Deep learning uses neural networks with multiple representation-learning layers. Early layers learn relatively simple patterns, while later layers combine them into more task-specific representations. For an image, the progression might be pixels → edges → textures → object parts → an object category.

| Traditional machine learning | Deep learning |
|---|---|
| Relies more heavily on manually designed features | Learns useful representations from data |
| Often effective with smaller, structured datasets | Often excels with large, unstructured datasets |
| Usually cheaper to train and easier to interpret | Can require substantial data and computation |
| Includes linear models, trees and boosting ensembles | Includes CNNs, RNNs and Transformers |

> Deep learning is not automatically the best choice. A simpler model may be preferable when the dataset is small, the inputs are well structured, interpretability is critical, or computational resources are limited.


### 1.2 Why deep learning became successful

The modern success of deep learning is often summarised through three mutually reinforcing factors.

- **Algorithms:** effective backpropagation, improved activation functions, better optimisation, dropout and pretraining made deeper networks easier to train.
- **Big data:** larger labelled and unlabelled datasets gave high-capacity models enough examples from which to learn.
- **Computing:** GPUs, specialised accelerators and distributed systems made large-scale matrix operations practical.

Geoffrey Hinton, Yann LeCun and Yoshua Bengio made foundational contributions to neural-network research. Hinton and colleagues' 2006 work on deep autoencoders helped renew interest in training multilayer networks, while the success of AlexNet in the 2012 ImageNet competition demonstrated the practical impact of deep convolutional networks.

:::: {.columns}
::: {.column width="48%"}
![Geoffrey Hinton at the 2024 Nobel Prize week in Stockholm](assets/geoffrey-hinton-2024.jpg){fig-alt="Geoffrey Hinton at the 2024 Nobel Prize week in Stockholm" style="height: 320px; width: auto; max-width: none;"}

**Geoffrey Hinton**

[Wikipedia profile](https://en.wikipedia.org/wiki/Geoffrey_Hinton) · [Image source and licence](https://commons.wikimedia.org/wiki/File:Geoffrey_E._Hinton,_2024_Nobel_Prize_Laureate_in_Physics.jpg)

Photograph by Arthur Petron, licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
:::

::: {.column width="48%"}
![Cover of the Deep Learning textbook published by MIT Press](assets/deep-learning-book-cover.jpg){fig-alt="Cover of the Deep Learning textbook by Ian Goodfellow, Yoshua Bengio and Aaron Courville" style="height: 320px; width: auto; max-width: none;"}

***Deep Learning* textbook**

Written by Ian Goodfellow, Yoshua Bengio and Aaron Courville—not by Hinton. Hinton provided a recommendation for the book.

[Free online edition](https://www.deeplearningbook.org/) · [MIT Press book page and cover source](https://mitpress.mit.edu/9780262337373/deep-learning/)
:::
::::



## 2. Deep Learning Architectures and Applications

The mind map organises this section from left to right. Deep learning is the common foundation on the left; the three application scenarios on the right are image processing with CNNs, sequence processing with RNNs, and language modelling with Transformers. Seq2Seq provides the conceptual bridge from recurrent sequence processing to attention-based models.

![Left-to-right deep learning mind map with a Deep Learning root and three application branches for images with CNNs, sequences with RNNs, and language models with Transformers](assets/session9-deep-learning-mind-map.svg){style="width: 900px; height: 495px; max-width: none;"}


### 2.1 Convolutional Neural Networks for Image Processing

#### 2.1.1 Why CNNs suit images

A colour image is commonly represented by a tensor with height, width and channel dimensions. A fully connected layer would assign a separate weight to every input-output connection, so the number of parameters grows quickly for large images. A **convolutional neural network (CNN)** reduces this burden through two ideas.

- **Local connectivity:** a filter examines a small neighbourhood rather than the whole image at once.
- **Weight sharing:** the same filter is applied at every spatial position.

A filter slides across the image and produces a **feature map**. During training, the network learns filters that respond to useful visual patterns. Activation functions add non-linearity, pooling can reduce spatial size, and deeper layers combine simple features into more complex ones. A common pattern is: convolution → activation → pooling → fully connected layer or task-specific output.

The next example applies a vertical-edge filter to a tiny greyscale image. This is a fixed filter rather than a trained CNN, but it demonstrates the local calculation performed by a convolutional layer.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use a small deterministic image so every convolution step can be inspected.
input_image = np.array(
    [
        [0, 0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
    ],
    dtype=float,
)

# Positive and negative columns make this kernel respond to vertical changes.
vertical_edge_kernel = np.array(
    [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]],
    dtype=float,
)

def valid_convolution(image, kernel):
    # Apply the kernel only where it fits fully inside the image.
    output_height = image.shape[0] - kernel.shape[0] + 1
    output_width = image.shape[1] - kernel.shape[1] + 1
    feature_map = np.zeros((output_height, output_width), dtype=float)

    for row in range(output_height):
        for column in range(output_width):
            image_patch = image[
                row : row + kernel.shape[0],
                column : column + kernel.shape[1],
            ]
            feature_map[row, column] = np.sum(image_patch * kernel)

    return feature_map

edge_feature_map = valid_convolution(input_image, vertical_edge_kernel)
print("Input shape:", input_image.shape)
print("Feature-map shape:", edge_feature_map.shape)
edge_feature_map


In [ ]:
# Use a shared layout so the image and feature map are easy to compare.
figure, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
axes[0].imshow(input_image, cmap="gray", interpolation="nearest")
axes[0].set_title("Input image")
axes[1].imshow(edge_feature_map, cmap="coolwarm", interpolation="nearest")
axes[1].set_title("Vertical-edge feature map")
for axis in axes:
    axis.set_xlabel("Column")
    axis.set_ylabel("Row")
plt.show()


Strong feature-map values occur where the dark and light regions meet. In a trained CNN, useful filters are learnt from examples rather than selected manually.

#### 2.1.2 CNNs for image classification

Image classification maps an entire image to one or more class labels. The final layer often produces class scores, which a softmax function can convert into probabilities for a single-label task. Cross-entropy loss compares those probabilities with the known label during training.

- **AlexNet** demonstrated the effectiveness of deep CNNs on large-scale image classification.
- **VGG** used a regular design with stacks of small convolutional filters.
- **Inception** processed information at multiple spatial scales.
- **ResNet** introduced residual connections that made very deep networks easier to optimise.

In practice, **transfer learning** is often more efficient than training from scratch. A pretrained network supplies general visual features, while a smaller task-specific output layer is trained for the new dataset. Accuracy, per-class recall and a confusion matrix reveal different aspects of performance.


#### 2.1.3 CNNs for object detection

Object detection answers two questions: **what objects are present, and where are they?** A detector normally produces class labels, confidence scores and bounding boxes for multiple objects.

| Task | Typical output | Common measure |
|---|---|---|
| Image classification | One label or label distribution for the image | Accuracy |
| Localisation | A label and one bounding box | Intersection over Union |
| Object detection | Multiple labels, boxes and confidence scores | mean Average Precision |

**Intersection over Union (IoU)** measures the overlap between a predicted box and a reference box. **mean Average Precision (mAP)** summarises precision-recall performance across classes and, depending on the benchmark, across one or more IoU thresholds.

- **Two-stage detectors**, such as the R-CNN family, first propose candidate regions and then classify and refine them.
- **One-stage detectors**, such as YOLO and SSD, predict classes and locations directly in a single detection pipeline.

One-stage designs are commonly associated with fast inference, while two-stage designs have historically prioritised localisation accuracy. The actual trade-off depends on the model version, input size, hardware and deployment constraints.


#### 2.1.4 CNNs for image generation

A discriminative model learns to separate or predict labels, whereas a generative model learns enough of a data distribution to produce new examples. Convolutional components are useful because generated images must preserve local spatial relationships.

A **Generative Adversarial Network (GAN)** contains two competing networks.

1. The **generator** converts a latent vector or conditioning input into a synthetic image.
2. The **discriminator** tries to distinguish real training images from generated images.
3. Adversarial training improves the generator until its outputs become harder to distinguish from real data.

DCGAN introduced convolutional design patterns for GANs; CycleGAN supports image-to-image translation; and SRGAN targets super-resolution. Modern diffusion systems often use a convolutional U-Net to predict and remove noise over a sequence of steps. Applications include synthesis, style transfer, restoration and super-resolution.

> Generative systems can reproduce dataset bias, create misleading content or memorise sensitive material. Data permission, copyright, provenance and human review therefore remain part of the technical workflow.



### 2.2 Recurrent Neural Networks for Speech and Text

#### 2.2.1 Sequence data and memory

Speech, text and time-series observations are ordered. The meaning of a word or sound may depend on what came before it, so treating every position as independent loses important context.

A **recurrent neural network (RNN)** processes one position at a time and maintains a hidden state. At time $t$, the state depends on the current input $x_t$ and the previous state $h_{t-1}$:

$$
h_t = \tanh(W_x x_t + W_h h_{t-1} + b)
$$

The same parameters are reused at every time step. This allows one network to process sequences of different lengths. Common patterns include many-to-one sentiment classification, one-to-many caption generation and many-to-many translation or sequence labelling.

The following scalar example shows how the current input and previous state jointly influence the next state. A real RNN uses vectors and matrices, but the recurrence is the same.


In [ ]:
# Positive values represent one signal; negative values represent an opposing signal.
sequence_inputs = np.array([0.8, 0.4, -0.2, 0.1, 0.9, -0.5], dtype=float)
input_weight = 1.1
recurrent_weight = 0.7
bias = -0.1
hidden_state = 0.0
hidden_history = []

for current_input in sequence_inputs:
    hidden_state = np.tanh(
        input_weight * current_input
        + recurrent_weight * hidden_state
        + bias
    )
    hidden_history.append(hidden_state)

for step, (current_input, state) in enumerate(
    zip(sequence_inputs, hidden_history),
    start=1,
):
    print(f"Step {step}: input={current_input:>4.1f}, hidden state={state:>6.3f}")


The state does not simply copy the current value: it combines new evidence with a compressed memory of previous inputs.

#### 2.2.2 Training challenges, LSTM and GRU

RNNs are trained using **backpropagation through time**, which unfolds the recurrent computation and propagates gradients across the sequence. Repeated multiplication can make gradients grow excessively or shrink towards zero. Gradient clipping can control exploding gradients, but ordinary RNNs may still struggle to learn long-term dependencies.

- **Long Short-Term Memory (LSTM)** networks use a cell state and input, forget and output gates to control information flow.
- **Gated Recurrent Units (GRUs)** combine some mechanisms into update and reset gates, producing a simpler recurrent unit.

LSTM and GRU models have been used for speech recognition, sentiment analysis, language modelling, captioning and translation. Neither is universally superior: the choice depends on the data, task, model size and training budget.


### 2.3 Transformers and Large Language Models

#### 2.3.1 Sequence-to-Sequence models

A **Sequence-to-Sequence (Seq2Seq)** model transforms one sequence into another. It is especially useful when input and output lengths differ. Early Seq2Seq systems commonly used RNNs or LSTMs in two components.

- The **encoder** reads the input sequence and constructs a semantic representation.
- The **decoder** uses that representation to generate an output sequence.

Machine translation, dialogue, text summarisation and speech recognition can all be expressed as sequence-to-sequence tasks. For example, an English sentence containing six tokens may translate into a sentence containing a different number of tokens.


#### 2.3.2 The fixed-context bottleneck and attention

A basic encoder compresses the complete input into one fixed-length context vector. This creates two related problems for long sequences.

1. One vector may not preserve every relevant detail from a long input.
2. A single representation does not explicitly indicate which input positions matter most for the current output.

**Attention** addresses this bottleneck by calculating a relevance score for each encoder state. After normalisation, the scores become weights used to create a context vector for the current decoding step. Different output steps can therefore focus on different parts of the input.

In query-key-value language, a **query** describes what the current step needs, **keys** describe the available positions, and **values** contain the information to combine. This small calculation gives three encoder positions different weights for one decoder query. It illustrates attention, not a trained translation model.


In [ ]:
source_tokens = ["deep", "learning", "works"]
encoder_keys = np.array(
    [[1.0, 0.1], [0.4, 1.2], [0.2, 0.8]],
    dtype=float,
)
decoder_query = np.array([0.3, 1.0], dtype=float)

# Dot products estimate how relevant each encoder position is to the query.
attention_scores = encoder_keys @ decoder_query
stable_scores = attention_scores - np.max(attention_scores)
attention_weights = np.exp(stable_scores) / np.exp(stable_scores).sum()

for token, score, weight in zip(
    source_tokens, attention_scores, attention_weights
):
    print(f"{token:>8}: score={score:.2f}, weight={weight:.3f}")

print("Weight total:", round(float(attention_weights.sum()), 6))


The weights sum to one, and the position most aligned with the query receives the greatest influence. In a real model, the representations and scoring parameters are learnt.

#### 2.3.3 From attention to Transformers and large language models

The Transformer replaced recurrence with attention-centred blocks. **Self-attention** lets every position compare with other positions in the same sequence, while **multi-head attention** learns several relationship patterns in parallel. Because attention alone does not encode order, positional information is added to token representations.

$$
\text{RNN encoder-decoder} \rightarrow \text{Seq2Seq with attention} \rightarrow \text{Transformer} \rightarrow \text{large language model}
$$

Transformer encoders are well suited to building contextual representations; BERT is a prominent encoder-oriented example. Autoregressive Transformer decoders predict the next token from preceding tokens; the GPT family follows this decoder-oriented pattern. Large language models extend the approach with large datasets, many parameters and extensive pretraining, followed by adaptation or instruction tuning.



## 3. Deep Learning Programming Frameworks

A deep learning framework provides tensor operations, automatic differentiation, reusable layers, loss functions, optimisers, data pipelines, accelerator support and model serialisation. These facilities allow practitioners to concentrate on the model and data rather than deriving and implementing every gradient manually.

### 3.1 PyTorch

PyTorch uses `torch.Tensor` for multidimensional data and automatic differentiation to record operations required for gradient calculation. Models commonly inherit from `torch.nn.Module`, while an explicit training loop makes the forward pass, loss calculation, gradient reset, backpropagation and optimiser step visible. This flexibility is useful for learning, research and custom model development.

```python
import torch
from torch import nn

model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(in_features=8 * 26 * 26, out_features=10),
)
```

The snippet is illustrative and is not executed in this notebook because PyTorch is an optional dependency.


### 3.2 TensorFlow and Keras

TensorFlow provides tensors, automatic differentiation and tools for training and deployment. Keras is its high-level model API. The `Sequential` API is concise for a simple stack of layers, while the Functional API supports models with branches, multiple inputs or shared layers. Built-in `compile`, `fit` and `evaluate` methods can manage a standard training workflow, and custom loops remain available when more control is required.

```python
import tensorflow as tf

model = tf.keras.Sequential(
    [
        tf.keras.layers.Conv2D(8, kernel_size=3, activation="relu"),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10),
    ]
)
```

This equivalent model definition is also illustrative and is not executed unless TensorFlow is installed.


### 3.3 Comparing the frameworks

| Area | PyTorch | TensorFlow with Keras |
|---|---|---|
| Tensor type | `torch.Tensor` | `tf.Tensor` |
| Model definition | `nn.Module` or `nn.Sequential` | Sequential, Functional or subclassed model |
| Standard training style | Often an explicit loop | Often `compile` and `fit` |
| Customisation | Direct, flexible Python control | High-level APIs plus custom loops |
| Deployment | PyTorch deployment and serving tools | Broad TensorFlow serving and edge ecosystem |

Neither framework is universally best. Consider the team's existing skills, pretrained-model availability, required deployment target, hardware support, maintainability and the degree of training-loop customisation required.

The environment check below does not import either framework. It reports whether each optional package is available, so the notebook remains runnable on a computer that has only NumPy and Matplotlib.


In [ ]:
import importlib.util

# find_spec checks package availability without loading a large framework.
framework_modules = {
    "PyTorch": "torch",
    "TensorFlow": "tensorflow",
}

for framework_name, module_name in framework_modules.items():
    is_available = importlib.util.find_spec(module_name) is not None
    status = "available" if is_available else "not installed"
    print(f"{framework_name}: {status}")


### 3.4 Apply your understanding

For each scenario, identify a suitable architecture and justify whether PyTorch or TensorFlow would be a reasonable implementation choice. More than one framework answer may be valid when supported by evidence.

1. Classify photographs of plant diseases using a modest labelled dataset.
2. Locate safety equipment in live worksite video.
3. Convert spoken customer enquiries into text.
4. Translate variable-length sentences from one language into another.
5. Generate new product images from text descriptions.

**Answer scaffold:**

- **Data type:** image, audio, text or another sequence
- **Task:** classification, detection, generation or sequence transformation
- **Architecture:** CNN, RNN/LSTM/GRU, Seq2Seq with attention, or Transformer
- **Framework evidence:** team experience, pretrained models, deployment target and required flexibility

**Suggested direction:** scenarios 1 and 2 point to CNN-based classification and detection; scenario 3 requires sequence modelling and is commonly handled by modern attention-based speech systems; scenario 4 is a Seq2Seq task now commonly implemented with a Transformer; scenario 5 is a generative vision task that may use a diffusion model with convolutional or Transformer components. Either framework can support these systems, so the framework decision requires project constraints rather than architecture name alone.

### 3.5 Key takeaways

- Deep learning learns layered representations and became practical through advances in algorithms, data and computing.
- CNNs exploit spatial structure for image classification, object detection and image generation.
- RNNs maintain sequence state; LSTM and GRU units help preserve longer dependencies.
- Seq2Seq maps one variable-length sequence to another, and attention reduces the fixed-context bottleneck.
- Transformers place attention at the centre of sequence modelling and underpin modern large language models.
- PyTorch and TensorFlow offer comparable foundations but different default workflows and ecosystems.

**Selected references:** LeCun, Bengio and Hinton (2015), *Deep learning*; Krizhevsky, Sutskever and Hinton (2012), *ImageNet Classification with Deep Convolutional Neural Networks*; Sutskever, Vinyals and Le (2014), *Sequence to Sequence Learning with Neural Networks*; Bahdanau, Cho and Bengio (2015), *Neural Machine Translation by Jointly Learning to Align and Translate*; Vaswani et al. (2017), *Attention Is All You Need*.
